# 성동구 일반음식점 생존분석 데이터 전처리

본 노트북은 **2021년 1분기(Q1)의 입지·상권 환경을 기준 변수로 사용**하고,
**2021-04-01부터 2026-09-12까지 음식점 폐업을 추적**하기 위한 음식점 단위 분석 데이터를 구축한다.

### 처리 순서
1. 일반음식점 인허가정보 로드
2. Baseline 시점 영업 음식점 추출
3. 서울시 서비스 업종 매핑
4. 좌표 정리 및 결측 좌표 지오코딩
5. 공식 상권 Spatial Join
6. 2021 Q1 점포·추정매출 데이터 정리
7. 상권×업종 점포당 추정매출 계산
8. 음식점에 상권 매출정보 연결
9. 매출환경 결측 보완
10. 임대료 변수 생성
11. 대중교통 변수 — 본 분석에서는 보류
12. 경쟁·집적 변수 생성
13. Survival 변수 생성
14. 음식점당 1행 최종 데이터 저장


## 0. 환경 설정


In [1]:
from pathlib import Path
import os
import re
import time
import unicodedata

import numpy as np
import pandas as pd
import geopandas as gpd
import requests

from pyproj import Transformer

# 데이터 폴더
DATA_DIR = Path("/Users/kim-eunhye/Downloads/캡스톤 데이터")

BASELINE_DATE = pd.Timestamp("2021-04-01")
STUDY_END_DATE = pd.Timestamp("2026-09-12")

def normalized(text):
    return unicodedata.normalize("NFC", str(text))

def find_file(*keywords, suffix=None):
    """DATA_DIR에서 파일명을 NFC 정규화한 뒤 키워드가 모두 포함된 파일 1개를 찾는다."""
    matches = []
    for p in DATA_DIR.iterdir():
        name = normalized(p.name)
        if suffix is not None and p.suffix.lower() != suffix.lower():
            continue
        if all(keyword in name for keyword in keywords):
            matches.append(p)

    if len(matches) != 1:
        raise FileNotFoundError(
            f"검색 조건 {keywords}, suffix={suffix}에 해당하는 파일이 1개여야 합니다. "
            f"현재 검색 결과: {[p.name for p in matches]}"
        )
    return matches[0]

print("DATA_DIR:", DATA_DIR)


DATA_DIR: /Users/kim-eunhye/Downloads/캡스톤 데이터


/Users/kim-eunhye/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 1. 일반음식점 인허가정보 → 기준 데이터


In [2]:
restaurant_path = find_file("성동구", "일반음식점", "인허가", suffix=".csv")
mapping_path = find_file("성동구_음식점_업태_서울상권서비스업종_매핑", suffix=".csv")

restaurant = pd.read_csv(
    restaurant_path,
    encoding="cp949",
    low_memory=False
)

mapping = pd.read_csv(
    mapping_path,
    encoding="utf-8-sig"
)

restaurant["인허가일자"] = pd.to_datetime(
    restaurant["인허가일자"], errors="coerce"
)
restaurant["폐업일자"] = pd.to_datetime(
    restaurant["폐업일자"], errors="coerce"
)

restaurant["restaurant_id"] = restaurant["관리번호"].astype(str)

print("인허가 데이터:", restaurant.shape)
print("Mapping Table:", mapping.shape)
print("관리번호 중복:", restaurant["restaurant_id"].duplicated().sum())
print("인허가일 결측:", restaurant["인허가일자"].isna().sum())


/var/folders/rw/wk8lm3652dxcbm3ss6qg0yj40000gn/T/ipykernel_45112/1735588731.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  restaurant["폐업일자"] = pd.to_datetime(


인허가 데이터: (15200, 40)
Mapping Table: (26, 6)
관리번호 중복: 0
인허가일 결측: 0


## 2. Baseline 시점 영업 음식점 추출

2021 Q1의 환경변수를 사용하므로 추적 시작일은 **2021-04-01**로 둔다.
이 단계에서는 baseline 시점에 영업 중이던 음식점만 선정하며,
`event`와 `duration`은 모든 설명변수를 만든 뒤 13절에서 생성한다.


In [3]:
baseline_df = restaurant[
    (restaurant["인허가일자"] <= BASELINE_DATE)
    &
    (
        restaurant["폐업일자"].isna()
        | (restaurant["폐업일자"] >= BASELINE_DATE)
    )
].copy()

print("전체 인허가 데이터:", len(restaurant))
print("Baseline 음식점:", len(baseline_df))
print("restaurant_id 중복:", baseline_df["restaurant_id"].duplicated().sum())
print(
    "Baseline 이전 폐업:",
    (
        baseline_df["폐업일자"].notna()
        & (baseline_df["폐업일자"] < BASELINE_DATE)
    ).sum()
)


전체 인허가 데이터: 15200
Baseline 음식점: 3787
restaurant_id 중복: 0
Baseline 이전 폐업: 0


## 3. 업태 Mapping Table 병합


In [4]:
baseline_df = baseline_df.merge(
    mapping,
    on="업태구분명",
    how="left",
    validate="many_to_one"
)

mapped = baseline_df["서울시_서비스_업종_코드"].notna()

print("병합 후 음식점:", len(baseline_df))
print("업종 매핑 성공:", mapped.sum())
print("업종 매핑 실패:", (~mapped).sum())
print("매핑 성공률:", round(mapped.mean() * 100, 2), "%")


병합 후 음식점: 3787
업종 매핑 성공: 3200
업종 매핑 실패: 587
매핑 성공률: 84.5 %


## 4. 음식점 좌표 정리


In [5]:
baseline_df["좌표정보(X)"] = pd.to_numeric(
    baseline_df["좌표정보(X)"], errors="coerce"
)
baseline_df["좌표정보(Y)"] = pd.to_numeric(
    baseline_df["좌표정보(Y)"], errors="coerce"
)

def clean_address(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    return np.nan if x == "" else x

baseline_df["도로명주소_clean"] = baseline_df["도로명주소"].apply(clean_address)
baseline_df["지번주소_clean"] = baseline_df["지번주소"].apply(clean_address)

missing_coord = baseline_df[
    baseline_df["좌표정보(X)"].isna()
    | baseline_df["좌표정보(Y)"].isna()
].copy()

print("Baseline 음식점:", len(baseline_df))
print("좌표 결측:", len(missing_coord))
print(
    "좌표 결측 중 주소 존재:",
    (
        missing_coord["도로명주소_clean"].notna()
        | missing_coord["지번주소_clean"].notna()
    ).sum()
)


Baseline 음식점: 3787
좌표 결측: 640
좌표 결측 중 주소 존재: 640


### 4-1. 결측 좌표 Kakao 지오코딩

API 키는 제출 파일에 직접 저장하지 않고 환경변수 `KAKAO_REST_API_KEY`에서 읽는다.
이미 원본 좌표가 있는 음식점에는 API를 호출하지 않는다.


In [6]:
def simplify_address(address):
    if address is None or pd.isna(address):
        return None

    address = str(address).strip()
    if address == "":
        return None

    address = re.sub(r"\s*\(.*$", "", address)
    address = address.split(",")[0].strip()
    return address


def geocode_kakao(address, api_key):
    if address is None or pd.isna(address):
        return None, None

    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {api_key}"}

    try:
        response = requests.get(
            url,
            headers=headers,
            params={"query": address},
            timeout=10
        )

        if response.status_code != 200:
            return None, None

        documents = response.json().get("documents", [])
        if not documents:
            return None, None

        return float(documents[0]["x"]), float(documents[0]["y"])

    except requests.RequestException:
        return None, None


def geocode_restaurant(row, api_key):
    candidates = [
        ("road_original", row["도로명주소_clean"]),
        ("road_simple", simplify_address(row["도로명주소_clean"])),
        ("jibun_original", row["지번주소_clean"]),
        ("jibun_simple", simplify_address(row["지번주소_clean"]))
    ]

    tried = set()

    for method, address in candidates:
        if address is None or pd.isna(address) or address in tried:
            continue

        tried.add(address)
        lon, lat = geocode_kakao(address, api_key)

        if lon is not None:
            return lon, lat, method, address

    return None, None, "failed", None


In [7]:
KAKAO_REST_API_KEY = "3056440a608048b06a0362da97a04e1e"

missing_indices = baseline_df[
    baseline_df["좌표정보(X)"].isna()
    | baseline_df["좌표정보(Y)"].isna()
].index

if len(missing_indices) > 0 and not KAKAO_REST_API_KEY:
    raise RuntimeError(
        "좌표 결측 음식점의 지오코딩을 위해 환경변수 "
        "'KAKAO_REST_API_KEY'를 설정한 뒤 다시 실행하세요."
    )

results = []

for count, idx in enumerate(missing_indices, start=1):
    row = baseline_df.loc[idx]

    lon, lat, method, used_address = geocode_restaurant(
        row, KAKAO_REST_API_KEY
    )

    results.append({
        "index": idx,
        "longitude": lon,
        "latitude": lat,
        "geocode_method": method,
        "geocode_used_address": used_address
    })

    if count % 100 == 0:
        print(f"{count} / {len(missing_indices)} 완료")

    time.sleep(0.05)

geocode_results = (
    pd.DataFrame(results).set_index("index")
    if results
    else pd.DataFrame(
        columns=[
            "longitude", "latitude",
            "geocode_method", "geocode_used_address"
        ]
    )
)


100 / 640 완료
200 / 640 완료
300 / 640 완료
400 / 640 완료
500 / 640 완료
600 / 640 완료


In [8]:
baseline_df["kakao_longitude"] = np.nan
baseline_df["kakao_latitude"] = np.nan
baseline_df["geocode_method"] = pd.NA
baseline_df["geocode_used_address"] = pd.NA

if len(geocode_results) > 0:
    baseline_df.loc[
        geocode_results.index, "kakao_longitude"
    ] = geocode_results["longitude"]

    baseline_df.loc[
        geocode_results.index, "kakao_latitude"
    ] = geocode_results["latitude"]

    baseline_df.loc[
        geocode_results.index, "geocode_method"
    ] = geocode_results["geocode_method"]

    baseline_df.loc[
        geocode_results.index, "geocode_used_address"
    ] = geocode_results["geocode_used_address"]

# 원자료 좌표는 EPSG:5174 → WGS84 변환
transformer = Transformer.from_crs(
    "EPSG:5174", "EPSG:4326", always_xy=True
)

has_original_coord = (
    baseline_df["좌표정보(X)"].notna()
    & baseline_df["좌표정보(Y)"].notna()
)

lon, lat = transformer.transform(
    baseline_df.loc[has_original_coord, "좌표정보(X)"].to_numpy(),
    baseline_df.loc[has_original_coord, "좌표정보(Y)"].to_numpy()
)

baseline_df.loc[has_original_coord, "original_longitude"] = lon
baseline_df.loc[has_original_coord, "original_latitude"] = lat

baseline_df["longitude"] = baseline_df["original_longitude"].fillna(
    baseline_df["kakao_longitude"]
)
baseline_df["latitude"] = baseline_df["original_latitude"].fillna(
    baseline_df["kakao_latitude"]
)

baseline_df["coordinate_source"] = np.select(
    [
        baseline_df["original_longitude"].notna(),
        baseline_df["kakao_longitude"].notna()
    ],
    ["original", "kakao_geocoded"],
    default="missing"
)

print(baseline_df["coordinate_source"].value_counts(dropna=False))
print("\n최종 좌표 결측:",
      (baseline_df["longitude"].isna() | baseline_df["latitude"].isna()).sum())


coordinate_source
original          3147
kakao_geocoded     632
missing              8
Name: count, dtype: int64

최종 좌표 결측: 8


## 5. 공식 상권코드 Spatial Join


In [9]:
shp_path = find_file("상권분석서비스", "영역-상권", suffix=".shp")
commercial = gpd.read_file(shp_path)

seongdong_commercial = commercial[
    commercial["SIGNGU_CD"].astype(str) == "11200"
].copy()

restaurant_geo = baseline_df[
    baseline_df["longitude"].notna()
    & baseline_df["latitude"].notna()
].copy()

restaurant_geo = gpd.GeoDataFrame(
    restaurant_geo,
    geometry=gpd.points_from_xy(
        restaurant_geo["longitude"],
        restaurant_geo["latitude"]
    ),
    crs="EPSG:4326"
).to_crs(commercial.crs)

restaurant_joined = gpd.sjoin(
    restaurant_geo,
    seongdong_commercial[
        [
            "TRDAR_SE_C",
            "TRDAR_SE_1",
            "TRDAR_CD",
            "TRDAR_CD_N",
            "geometry"
        ]
    ],
    how="left",
    predicate="within"
)

print("전체 상권:", len(commercial))
print("성동구 상권:", len(seongdong_commercial))
print("좌표 보유 음식점:", len(restaurant_geo))
print("공식 상권 내부:", restaurant_joined["TRDAR_CD"].notna().sum())
print("공식 상권 밖:", restaurant_joined["TRDAR_CD"].isna().sum())
print("restaurant_id 중복:",
      restaurant_joined["restaurant_id"].duplicated().sum())


전체 상권: 1650
성동구 상권: 55
좌표 보유 음식점: 3779
공식 상권 내부: 2910
공식 상권 밖: 869
restaurant_id 중복: 0


### 5-1. 공식 상권 밖 음식점 거리 진단

상권 밖 음식점은 임의로 공식 상권 내부로 편입하지 않는다.
다만 가장 가까운 공식 상권까지의 거리를 진단용으로 계산한다.


In [10]:
outside = restaurant_joined[
    restaurant_joined["TRDAR_CD"].isna()
].drop(columns=["index_right"], errors="ignore").copy()

nearest = gpd.sjoin_nearest(
    outside,
    seongdong_commercial[
        ["TRDAR_CD", "TRDAR_CD_N", "TRDAR_SE_1", "geometry"]
    ],
    how="left",
    distance_col="distance_to_market"
)

nearest_unique = (
    nearest
    .sort_values(["restaurant_id", "distance_to_market"])
    .drop_duplicates("restaurant_id", keep="first")
    .copy()
)

print("공식 상권 밖:", len(outside))
print("최근접 결과(중복 제거):", len(nearest_unique))
print(nearest_unique["distance_to_market"].describe())


공식 상권 밖: 869
최근접 결과(중복 제거): 869
count    869.000000
mean     126.654231
std      152.230137
min        1.804706
25%       24.735828
50%       56.348948
75%      199.772765
max      727.569250
Name: distance_to_market, dtype: float64


## 6. 2021 Q1 점포-상권 + 추정매출-상권 병합


In [11]:
store_path = find_file("점포-상권", "2021", suffix=".csv")
sales_path = find_file("추정매출-상권", "2021", suffix=".csv")

store_df = pd.read_csv(
    store_path, encoding="cp949", low_memory=False
)
sales_df = pd.read_csv(
    sales_path, encoding="cp949", low_memory=False
)

store_q1 = store_df[
    store_df["기준_년분기_코드"] == 20211
].copy()
sales_q1 = sales_df[
    sales_df["기준_년분기_코드"] == 20211
].copy()

food_codes = [
    "CS100001", "CS100002", "CS100003", "CS100004", "CS100005",
    "CS100006", "CS100007", "CS100008", "CS100009", "CS100010"
]

seongdong_market_codes = set(
    seongdong_commercial["TRDAR_CD"].astype(str)
)

for df in [store_q1, sales_q1]:
    df["상권_코드"] = df["상권_코드"].astype(str)
    df["서비스_업종_코드"] = df["서비스_업종_코드"].astype(str)

store_sd_food = store_q1[
    store_q1["상권_코드"].isin(seongdong_market_codes)
    & store_q1["서비스_업종_코드"].isin(food_codes)
].copy()

sales_sd_food = sales_q1[
    sales_q1["상권_코드"].isin(seongdong_market_codes)
    & sales_q1["서비스_업종_코드"].isin(food_codes)
].copy()

print("2021 Q1 점포:", store_sd_food.shape)
print("2021 Q1 추정매출:", sales_sd_food.shape)
print("점포 key 중복:",
      store_sd_food.duplicated(["상권_코드", "서비스_업종_코드"]).sum())
print("매출 key 중복:",
      sales_sd_food.duplicated(["상권_코드", "서비스_업종_코드"]).sum())


2021 Q1 점포: (423, 14)
2021 Q1 추정매출: (216, 55)
점포 key 중복: 0
매출 key 중복: 0


## 7. 상권×업종 점포당 추정매출 계산


In [12]:
store_columns = [
    "상권_코드", "상권_코드_명",
    "서비스_업종_코드", "서비스_업종_코드_명",
    "점포_수", "유사_업종_점포_수",
    "개업_율", "개업_점포_수",
    "폐업_률", "폐업_점포_수",
    "프랜차이즈_점포_수"
]

sales_columns = [
    "상권_코드", "서비스_업종_코드",
    "당월_매출_금액", "당월_매출_건수"
]

area_q1 = (
    store_sd_food[store_columns]
    .merge(
        sales_sd_food[sales_columns],
        on=["상권_코드", "서비스_업종_코드"],
        how="left",
        validate="one_to_one"
    )
)

area_q1["area_category_sales_per_store"] = np.where(
    area_q1["점포_수"] > 0,
    area_q1["당월_매출_금액"] / area_q1["점포_수"],
    np.nan
)

print("상권×업종 조합:", len(area_q1))
print(
    "점포당 추정매출 관측:",
    area_q1["area_category_sales_per_store"].notna().sum()
)


상권×업종 조합: 423
점포당 추정매출 관측: 209


## 8. 음식점에 상권 매출정보 연결


In [13]:
restaurant_joined["TRDAR_CD"] = (
    restaurant_joined["TRDAR_CD"].astype("string")
)
restaurant_joined["서울시_서비스_업종_코드"] = (
    restaurant_joined["서울시_서비스_업종_코드"].astype("string")
)

area_q1["상권_코드"] = area_q1["상권_코드"].astype("string")
area_q1["서비스_업종_코드"] = (
    area_q1["서비스_업종_코드"].astype("string")
)

area_variables = area_q1[
    [
        "상권_코드", "서비스_업종_코드",
        "점포_수", "유사_업종_점포_수",
        "개업_율", "개업_점포_수",
        "폐업_률", "폐업_점포_수",
        "프랜차이즈_점포_수",
        "당월_매출_금액", "당월_매출_건수",
        "area_category_sales_per_store"
    ]
].copy()

area_variables = area_variables.rename(columns={
    "상권_코드": "TRDAR_CD",
    "서비스_업종_코드": "서울시_서비스_업종_코드",
    "점포_수": "area_category_store_count",
    "유사_업종_점포_수": "area_similar_store_count",
    "개업_율": "area_open_rate",
    "개업_점포_수": "area_open_count",
    "폐업_률": "area_close_rate",
    "폐업_점포_수": "area_close_count",
    "프랜차이즈_점포_수": "area_franchise_count",
    "당월_매출_금액": "area_category_sales",
    "당월_매출_건수": "area_category_sales_count"
})

restaurant_merged = restaurant_joined.merge(
    area_variables,
    on=["TRDAR_CD", "서울시_서비스_업종_코드"],
    how="left",
    validate="many_to_one"
)

inside = restaurant_merged["TRDAR_CD"].notna()
mapped = restaurant_merged["서울시_서비스_업종_코드"].notna()
direct_sales = restaurant_merged["area_category_sales_per_store"].notna()

print("음식점:", len(restaurant_merged))
print("공식 상권 내부:", inside.sum())
print("업종 매핑 성공:", mapped.sum())
print("Direct 매출환경 관측:", direct_sales.sum())


음식점: 3779
공식 상권 내부: 2910
업종 매핑 성공: 3192
Direct 매출환경 관측: 2139


## 9. 매출환경 결측 보완

공식 상권 내부이고 업종 매핑이 되었지만 해당 상권×업종의 매출이 없는 경우에만
**동일 업종의 매출 관측 상권**을 최대 1km 안에서 탐색한다.

- 참조 상권 3개 이상: 가까운 3개 IDW
- 참조 상권 2개: 2개 IDW
- 참조 상권 1개: 최근접 1개
- 1km 이내 참조 없음: 결측 유지

공식 상권 밖 또는 업종 미매핑 음식점은 매출환경을 임의 추정하지 않는다.


In [14]:
# 성동구 55개 상권의 centroid
market_centroids = seongdong_commercial[
    ["TRDAR_CD", "geometry"]
].copy()

market_centroids["TRDAR_CD"] = (
    market_centroids["TRDAR_CD"].astype("string")
)
market_centroids["geometry"] = market_centroids.geometry.centroid

# 실제 점포당 매출이 관측된 상권×업종만 참조 후보로 사용
sales_reference = area_variables[
    area_variables["area_category_sales_per_store"].notna()
].merge(
    market_centroids,
    on="TRDAR_CD",
    how="left",
    validate="many_to_one"
)

sales_reference = gpd.GeoDataFrame(
    sales_reference,
    geometry="geometry",
    crs=seongdong_commercial.crs
)

restaurant_merged["area_category_sales_per_store_final"] = (
    restaurant_merged["area_category_sales_per_store"]
)
restaurant_merged["sales_source"] = np.where(
    restaurant_merged["area_category_sales_per_store"].notna(),
    "direct",
    pd.NA
)

missing_mask = (
    restaurant_merged["TRDAR_CD"].notna()
    & restaurant_merged["서울시_서비스_업종_코드"].notna()
    & restaurant_merged["area_category_sales_per_store"].isna()
)

for idx, row in restaurant_merged.loc[missing_mask].iterrows():
    category = row["서울시_서비스_업종_코드"]

    refs = sales_reference[
        sales_reference["서울시_서비스_업종_코드"] == category
    ].copy()

    if refs.empty:
        continue

    distances = refs.geometry.distance(row.geometry)
    refs = refs.assign(distance=distances).query("distance <= 1000")
    refs = refs.sort_values("distance").head(3)

    if len(refs) == 0:
        continue

    values = refs["area_category_sales_per_store"].to_numpy(dtype=float)
    d = refs["distance"].to_numpy(dtype=float)

    # 동일 위치가 있으면 그 값을 그대로 사용
    if np.any(d == 0):
        estimate = values[np.argmin(d)]
    elif len(refs) >= 2:
        weights = 1 / d
        estimate = np.average(values, weights=weights)
    else:
        estimate = values[0]

    if len(refs) >= 3:
        method = "idw_3"
    elif len(refs) == 2:
        method = "idw_2"
    else:
        method = "nearest_1"

    restaurant_merged.at[
        idx, "area_category_sales_per_store_final"
    ] = estimate
    restaurant_merged.at[idx, "sales_source"] = method

restaurant_merged.loc[
    missing_mask
    & restaurant_merged["area_category_sales_per_store_final"].isna(),
    "sales_source"
] = "unavailable"

print(
    restaurant_merged.loc[missing_mask, "sales_source"]
    .value_counts(dropna=False)
)
print(
    "최종 매출환경 관측:",
    restaurant_merged["area_category_sales_per_store_final"]
    .notna().sum()
)


sales_source
idw_3          179
nearest_1       78
idw_2           63
unavailable      4
Name: count, dtype: int64
최종 매출환경 관측: 2459


In [15]:
# 업종별 direct 매출 분포의 20/40/60/80 분위수를 기준으로 5단계 구간 생성
quantiles = (
    area_q1
    .dropna(subset=["area_category_sales_per_store"])
    .groupby("서비스_업종_코드")["area_category_sales_per_store"]
    .quantile([0.2, 0.4, 0.6, 0.8])
    .unstack()
)

quantiles.columns = ["q20", "q40", "q60", "q80"]

def classify_sales_level(row):
    value = row["area_category_sales_per_store_final"]
    category = row["서울시_서비스_업종_코드"]

    if pd.isna(value) or pd.isna(category) or category not in quantiles.index:
        return pd.NA

    q20, q40, q60, q80 = quantiles.loc[
        category, ["q20", "q40", "q60", "q80"]
    ]

    if value <= q20:
        return "하"
    if value <= q40:
        return "중하"
    if value <= q60:
        return "중"
    if value <= q80:
        return "중상"
    return "상"

restaurant_merged["sales_level_5"] = restaurant_merged.apply(
    classify_sales_level,
    axis=1
)

restaurant_merged["sales_missing_reason"] = "available"

restaurant_merged.loc[
    restaurant_merged["TRDAR_CD"].isna(),
    "sales_missing_reason"
] = "outside_commercial_area"

restaurant_merged.loc[
    restaurant_merged["서울시_서비스_업종_코드"].isna(),
    "sales_missing_reason"
] = "category_unmapped"

restaurant_merged.loc[
    restaurant_merged["TRDAR_CD"].notna()
    & restaurant_merged["서울시_서비스_업종_코드"].notna()
    & restaurant_merged["area_category_sales_per_store_final"].isna(),
    "sales_missing_reason"
] = "no_reference_within_1km"

print(restaurant_merged["sales_missing_reason"].value_counts())
print("\n매출수준:")
print(restaurant_merged["sales_level_5"].value_counts(dropna=False))


sales_missing_reason
available                  2459
outside_commercial_area     729
category_unmapped           587
no_reference_within_1km       4
Name: count, dtype: int64

매출수준:
sales_level_5
<NA>    1320
중        620
중상       604
상        455
중하       423
하        357
Name: count, dtype: int64


## 10. 임대료 변수

교수님 피드백에 따라 임대료는 핵심 통제변수로 사용한다.
2021 Q1의 성동구 임대료 자료를 **동 단위로 집계**하여 음식점 주소의 법정동과 연결한다.
연속형 평균 임대료와 함께 해석이 쉬운 **상·중·하 3단계 변수**도 생성한다.


In [16]:
rent_path = find_file("성동구", "상가임대료", suffix=".csv")

rent = pd.read_csv(
    rent_path,
    encoding="cp949",
    low_memory=False
)

rent["기준년월"] = pd.to_datetime(
    rent["기준년월"], errors="coerce"
)
rent["평당전세가"] = pd.to_numeric(
    rent["평당전세가"], errors="coerce"
)

rent_q1 = rent[
    rent["기준년월"].between(
        pd.Timestamp("2021-01-01"),
        pd.Timestamp("2021-03-31")
    )
].copy()

rent_dong_summary = (
    rent_q1
    .groupby("읍면동")["평당전세가"]
    .agg(
        rent_mean="mean",
        rent_median="median",
        rent_count="count"
    )
    .reset_index()
    .rename(columns={"읍면동": "rent_dong"})
)

# 동별 평균 임대료를 동일 가중으로 3분위 분류
rent_dong_summary["rent_level_3"] = pd.qcut(
    rent_dong_summary["rent_mean"].rank(method="first"),
    q=3,
    labels=["하", "중", "상"]
)

def extract_dong(address):
    if pd.isna(address):
        return pd.NA
    match = re.search(
        r"([가-힣]+(?:동\d*가|동|가))(?=\s|\d|-|$)",
        str(address)
    )
    return match.group(1) if match else pd.NA

restaurant_merged["rent_dong"] = (
    restaurant_merged["지번주소"].apply(extract_dong)
)

restaurant_merged = restaurant_merged.merge(
    rent_dong_summary,
    on="rent_dong",
    how="left",
    validate="many_to_one"
)

print("임대료 매칭 성공:", restaurant_merged["rent_mean"].notna().sum())
print("임대료 매칭 실패:", restaurant_merged["rent_mean"].isna().sum())
print(restaurant_merged["rent_level_3"].value_counts(dropna=False))


임대료 매칭 성공: 3779
임대료 매칭 실패: 0
rent_level_3
하    1956
상    1194
중     629
Name: count, dtype: int64


## 11. 지하철/버스 반경 변수

대중교통 접근성 변수는 추가 데이터 검토 후 확장할 수 있도록 본 제출본에서는 보류한다.
현재 분석에서는 임대료가 입지의 종합적인 시장가치를 반영하는 핵심 통제변수라는 교수님 피드백을 우선 반영한다.


## 12. 경쟁 및 집적 변수

각 음식점을 중심으로 300m와 500m 반경 안의 음식점 수를 계산한다.

- `restaurants_all_*`: 주변 전체 음식점 수
- `competitors_same_*`: 동일 서울시 서비스 업종 음식점 수

자기 자신은 경쟁 음식점 수에서 제외한다.


In [17]:
# restaurant_merged는 이미 EPSG:5181 GeoDataFrame
points = restaurant_merged[
    ["restaurant_id", "서울시_서비스_업종_코드", "geometry"]
].copy()

points = gpd.GeoDataFrame(
    points,
    geometry="geometry",
    crs=restaurant_merged.crs
)

left = points[
    ["restaurant_id", "서울시_서비스_업종_코드", "geometry"]
].rename(columns={
    "restaurant_id": "restaurant_id_left",
    "서울시_서비스_업종_코드": "category_left"
})

right = points[
    ["restaurant_id", "서울시_서비스_업종_코드", "geometry"]
].rename(columns={
    "restaurant_id": "restaurant_id_right",
    "서울시_서비스_업종_코드": "category_right"
})

# 최대 반경 500m 후보 쌍 생성
buffers = left.copy()
buffers["geometry"] = buffers.geometry.buffer(500)

pairs = gpd.sjoin(
    buffers,
    right,
    how="left",
    predicate="intersects"
)

# 자기 자신 제외
pairs = pairs[
    pairs["restaurant_id_left"] != pairs["restaurant_id_right"]
].copy()

# 실제 점-점 거리 계산
left_geom = points.set_index("restaurant_id")["geometry"]
right_geom = points.set_index("restaurant_id")["geometry"]

pairs["distance_m"] = [
    left_geom.loc[lid].distance(right_geom.loc[rid])
    for lid, rid in zip(
        pairs["restaurant_id_left"],
        pairs["restaurant_id_right"]
    )
]

pairs["same_category"] = (
    pairs["category_left"].notna()
    & (pairs["category_left"] == pairs["category_right"])
)

def neighbor_counts(radius):
    subset = pairs[pairs["distance_m"] <= radius]

    all_count = (
        subset.groupby("restaurant_id_left")
        .size()
        .reindex(points["restaurant_id"], fill_value=0)
    )

    same_count = (
        subset[subset["same_category"]]
        .groupby("restaurant_id_left")
        .size()
        .reindex(points["restaurant_id"], fill_value=0)
    )

    return all_count, same_count

all_300, same_300 = neighbor_counts(300)
all_500, same_500 = neighbor_counts(500)

restaurant_merged["restaurants_all_300m"] = (
    restaurant_merged["restaurant_id"].map(all_300).astype(int)
)
restaurant_merged["restaurants_all_500m"] = (
    restaurant_merged["restaurant_id"].map(all_500).astype(int)
)

restaurant_merged["competitors_same_300m"] = (
    restaurant_merged["restaurant_id"].map(same_300).astype("Int64")
)
restaurant_merged["competitors_same_500m"] = (
    restaurant_merged["restaurant_id"].map(same_500).astype("Int64")
)

# 업종 자체가 매핑되지 않은 음식점은 동일업종 경쟁 변수 정의 불가
unmapped = restaurant_merged["서울시_서비스_업종_코드"].isna()
restaurant_merged.loc[
    unmapped,
    ["competitors_same_300m", "competitors_same_500m"]
] = pd.NA

print(
    restaurant_merged[
        [
            "restaurants_all_300m",
            "competitors_same_300m",
            "restaurants_all_500m",
            "competitors_same_500m"
        ]
    ].describe()
)

print(
    "\n500m < 300m 오류:",
    (
        restaurant_merged["restaurants_all_500m"]
        < restaurant_merged["restaurants_all_300m"]
    ).sum()
)


       restaurants_all_300m  competitors_same_300m  restaurants_all_500m  \
count           3779.000000                 3192.0           3779.000000   
mean             164.958984              51.089599            327.688277   
std               91.495829              47.687759            160.577777   
min                1.000000                    0.0             11.000000   
25%               85.000000                   10.0            166.000000   
50%              155.000000                   33.0            346.000000   
75%              235.000000                  90.25            465.000000   
max              366.000000                  178.0            724.000000   

       competitors_same_500m  
count                 3192.0  
mean              100.015664  
std                90.180611  
min                      0.0  
25%                     21.0  
50%                     65.0  
75%                    178.0  
max                    372.0  

500m < 300m 오류: 0


## 13. Duration, Event 생성

- 추적 시작: 2021-04-01
- 관찰 종료: 2026-09-12
- 관찰기간 중 폐업: `event = 1`
- 관찰 종료일까지 폐업 미관측: `event = 0`
- `duration`: baseline부터 폐업일 또는 관찰종료일까지의 기간
- `business_age_at_baseline`: baseline 이전에 이미 영업한 기간


In [18]:
restaurant_merged["인허가일자"] = pd.to_datetime(
    restaurant_merged["인허가일자"], errors="coerce"
)
restaurant_merged["폐업일자"] = pd.to_datetime(
    restaurant_merged["폐업일자"], errors="coerce"
)

restaurant_merged["event"] = (
    restaurant_merged["폐업일자"].notna()
    & (restaurant_merged["폐업일자"] >= BASELINE_DATE)
    & (restaurant_merged["폐업일자"] <= STUDY_END_DATE)
).astype(int)

restaurant_merged["observation_end"] = (
    restaurant_merged["폐업일자"]
    .where(
        restaurant_merged["event"] == 1,
        STUDY_END_DATE
    )
)

restaurant_merged["duration_days"] = (
    restaurant_merged["observation_end"] - BASELINE_DATE
).dt.days

restaurant_merged["duration_months"] = (
    restaurant_merged["duration_days"] / 30.4375
)

restaurant_merged["business_age_at_baseline_years"] = (
    (BASELINE_DATE - restaurant_merged["인허가일자"]).dt.days
    / 365.25
)

print("분석 대상 음식점:", len(restaurant_merged))
print("\nEvent:")
print(restaurant_merged["event"].value_counts().sort_index())
print("\nDuration:")
print(
    restaurant_merged[
        ["duration_days", "duration_months"]
    ].describe()
)
print("\n음수 duration:",
      (restaurant_merged["duration_days"] < 0).sum())
print("duration 결측:",
      restaurant_merged["duration_days"].isna().sum())


분석 대상 음식점: 3779

Event:
event
0    2119
1    1660
Name: count, dtype: int64

Duration:
       duration_days  duration_months
count    3779.000000      3779.000000
mean     1495.637470        49.137987
std       674.515529        22.160674
min         0.000000         0.000000
25%       942.000000        30.948665
50%      1990.000000        65.379877
75%      1990.000000        65.379877
max      1990.000000        65.379877

음수 duration: 0
duration 결측: 0


In [19]:
# Cox 모델에서 duration=0인 관측치가 있는지 별도 확인
zero_duration = restaurant_merged[
    restaurant_merged["duration_days"] == 0
]

print("duration=0 음식점:", len(zero_duration))

if len(zero_duration) > 0:
    display(
        zero_duration[
            [
                "restaurant_id",
                "사업장명",
                "인허가일자",
                "폐업일자",
                "event",
                "duration_days"
            ]
        ]
    )


duration=0 음식점: 4


,restaurant_id,사업장명,인허가일자,폐업일자,event,duration_days
926,3030000-101-2014-00096,이상진신닭발,2014-04-14,2021-04-01,1,0
1071,3030000-101-2019-00274,비비해이 BBHAY,2019-08-01,2021-04-01,1,0
1926,3030000-101-1995-01629,무한리필 고기굼터,1995-03-15,2021-04-01,1,0
3157,3030000-101-2020-00284,집밥으로 우주정복,2020-07-13,2021-04-01,1,0


## 14. 최종 음식점당 1행 저장


In [20]:
final_columns = [
    # 식별 정보
    "restaurant_id",
    "사업장명",
    "인허가일자",
    "폐업일자",
    "업태구분명",

    # 위치
    "도로명주소",
    "지번주소",
    "longitude",
    "latitude",

    # 업종
    "서울시_서비스_업종_코드",
    "서울시_서비스_업종_코드_명",

    # 공식 상권
    "TRDAR_CD",
    "TRDAR_CD_N",
    "TRDAR_SE_1",

    # 매출환경: direct 원값 + 보완된 최종값을 모두 보존
    "area_category_sales_per_store",
    "area_category_sales_per_store_final",
    "sales_source",
    "sales_level_5",
    "sales_missing_reason",

    # 임대료
    "rent_dong",
    "rent_mean",
    "rent_median",
    "rent_count",
    "rent_level_3",

    # 경쟁·집적
    "restaurants_all_300m",
    "competitors_same_300m",
    "restaurants_all_500m",
    "competitors_same_500m",

    # Survival
    "business_age_at_baseline_years",
    "event",
    "observation_end",
    "duration_days",
    "duration_months"
]

final_df = restaurant_merged[final_columns].copy()

print("최종 데이터 shape:", final_df.shape)
print("고유 restaurant_id:", final_df["restaurant_id"].nunique())
print("restaurant_id 중복:",
      final_df["restaurant_id"].duplicated().sum())


최종 데이터 shape: (3779, 33)
고유 restaurant_id: 3779
restaurant_id 중복: 0


In [21]:
check_columns = [
    "서울시_서비스_업종_코드",
    "TRDAR_CD",
    "area_category_sales_per_store_final",
    "rent_mean",
    "restaurants_all_300m",
    "competitors_same_300m",
    "duration_days"
]

missing_summary = pd.DataFrame({
    "missing_count": final_df[check_columns].isna().sum(),
    "missing_percent": (
        final_df[check_columns]
        .isna()
        .mean()
        .mul(100)
        .round(2)
    )
})

display(missing_summary)

assert final_df["restaurant_id"].duplicated().sum() == 0
assert final_df["duration_days"].isna().sum() == 0
assert (final_df["duration_days"] < 0).sum() == 0
assert final_df["rent_mean"].isna().sum() == 0
assert final_df["restaurants_all_300m"].isna().sum() == 0

print("최종 검증 완료")


,missing_count,missing_percent
서울시_서비스_업종_코드,587,15.53
TRDAR_CD,869,23.00
area_category_sales_per_store_final,1320,34.93
rent_mean,0,0.00
restaurants_all_300m,0,0.00
competitors_same_300m,587,15.53
duration_days,0,0.00


최종 검증 완료


In [22]:
output_path = DATA_DIR / "final_restaurant_level_dataset.csv"

final_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", output_path)


저장 완료: /Users/kim-eunhye/Downloads/캡스톤 데이터/final_restaurant_level_dataset.csv
